In [1]:
import pandas as pd
import requests

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
df = pd.read_csv("data/processed/nyc311_ida_labeled.csv")

NEAR_MISS = ["Water System", "Street Condition", "Damaged Tree", "Sewer", "Standing Water",
             "Traffic Signal Condition", "Street Light Condition",
             "Overgrown Tree/Branches", "Dead/Dying Tree"]

flood = df[df["label"] == "flood"].sample(40, random_state=1)
road = df[df["label"] == "road_blockage"].sample(20, random_state=1)
non = df[(df["label"] == "non_hazard") & (df["complaint_type"].isin(NEAR_MISS))].sample(40, random_state=1)
chk = pd.concat([flood, road, non])
print(chk["label"].value_counts())

label
flood            40
non_hazard       40
road_blockage    20
Name: count, dtype: int64


In [2]:
keys = ", ".join(f"'{k}'" for k in chk["unique_key"])
r = requests.get(URL, params={
    "$where": f"unique_key in ({keys})",
    "$select": "unique_key, resolution_description",
    "$limit": 200}, timeout=60)
r.raise_for_status()
res = pd.DataFrame(r.json())
res["unique_key"] = res["unique_key"].astype("int64")

chk = chk.merge(res, on="unique_key", how="left")
out = chk[["unique_key", "label", "complaint_type", "descriptor",
           "location_type", "resolution_description"]].copy()
out["my_verdict"] = ""
out.to_csv("data/processed/label_check_311.csv", index=False, encoding="utf-8-sig")
print(len(out), "rows saved to data/processed/label_check_311.csv")
out["resolution_description"].notna().mean()

100 rows saved to data/processed/label_check_311.csv


np.float64(0.99)

In [8]:
v = pd.read_csv("data/processed/label_check_311.csv", encoding="utf-8-sig")
v["my_verdict"] = v["my_verdict"].astype(str).str.strip().str.lower()

empty = (v["my_verdict"].isin(["nan", ""])).sum()
assert empty == 0, f"{empty} rows still have no verdict - save the full file first"

print(pd.crosstab(v["label"], v["my_verdict"], margins=True))

known = v[v["my_verdict"].isin(["ok", "wrong"])]
print("\nError rate (wrong / judged, 'unclear' excluded):")
print((known.groupby("label")["my_verdict"].apply(lambda s: (s == "wrong").mean())).round(2))

my_verdict     ok  unclear  wrong  All
label                                 
flood          26       10      4   40
non_hazard     29        0     11   40
road_blockage  16        1      3   20
All            71       11     18  100

Error rate (wrong / judged, 'unclear' excluded):
label
flood            0.13
non_hazard       0.28
road_blockage    0.16
Name: my_verdict, dtype: float64


In [9]:
pd.set_option("display.max_colwidth", 100)
bad = v[v["my_verdict"].isin(["wrong", "unclear"])]
print(bad.groupby(["label", "complaint_type", "descriptor", "my_verdict"]).size()
         .sort_values(ascending=False).to_string())

label          complaint_type            descriptor                               my_verdict
flood          Sewer                     Sewer Backup (Use Comments) (SA)         unclear       7
non_hazard     Traffic Signal Condition  Controller                               wrong         6
flood          Sewer                     Sewer Backup (Use Comments) (SA)         wrong         3
               Water System              Excessive Water In Basement (WEFB)       unclear       2
non_hazard     Sewer                     Manhole Cover Missing (Emergency) (SA3)  wrong         2
flood          Sewer                     Manhole Overflow (Use Comments) (SA1)    wrong         1
non_hazard     Overgrown Tree/Branches   Hitting Power Lines                      wrong         1
flood          Sewer                     Street Flooding (SJ)                     unclear       1
non_hazard     Street Light Condition    Lamppost Knocked Down                    wrong         1
               Traffic Si

In [10]:
sub = v[v["descriptor"].str.contains("Sewer Backup|Controller|Manhole Cover Missing|Tree Has Fallen|Branch or Limb|Leaning", regex=True)]
print(pd.crosstab([sub["label"], sub["descriptor"]], sub["my_verdict"], margins=True))

my_verdict                                             ok  unclear  wrong  All
label         descriptor                                                      
flood         Sewer Backup (Use Comments) (SA)         19        7      3   29
non_hazard    Controller                                0        0      6    6
              Manhole Cover Missing (Emergency) (SA3)   0        0      2    2
road_blockage Branch or Limb Has Fallen Down            8        1      1   10
              Entire Tree Has Fallen Down               4        0      1    5
              Tree Leaning/Uprooted                     0        0      1    1
All                                                    31        8     14   53


In [11]:
pd.set_option("display.max_colwidth", 260)
ctrl = v[v["descriptor"] == "Controller"]
display(ctrl[["my_verdict", "location_type", "resolution_description"]])

sb = v[v["descriptor"].str.startswith("Sewer Backup")]
print(pd.crosstab(sb["location_type"], sb["my_verdict"]))

,my_verdict,location_type,resolution_description
70,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.
73,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.
74,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.
77,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.
87,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.
89,wrong,NaN,Service Request status for this request is available on the Department of Transportationâs website. Please click the âLearn Moreâ link below.


Empty DataFrame
Columns: []
Index: []


In [7]:
import os, time
import pandas as pd

p = "data/processed/label_check_311.csv"
print("last saved:", time.ctime(os.path.getmtime(p)))
v = pd.read_csv(p, encoding="utf-8-sig")
print(v.shape)
print(v["my_verdict"].value_counts(dropna=False))

last saved: Thu Oct  8 10:59:18 2026
(100, 7)
my_verdict
ok         46
ok         25
wrong      18
unclear    11
Name: count, dtype: int64


## Findings

- Manual check of 100 weak labels (40 flood, 20 road_blockage, 40 non_hazard sampled from near-miss complaint types), one reviewer.
- flood: 26 ok / 10 unclear / 4 wrong; road_blockage: 16 ok / 1 unclear / 3 wrong; non_hazard (near-miss types only): 29 ok / 11 wrong.
- Sewer Backup (69% of flood labels): about 66% confirmed (19 ok / 7 unclear / 3 wrong of 29 sampled).
- Traffic Signal "Controller": 6 of 6 judged hazards by the reviewer but discarded by the rules. Evidence is weak (resolution notes are a generic DOT redirect), so it is not added to the rules.
- Limits: small samples, one reviewer, subjective judgments.